
# Major Index ML Top-5 Branch Comparison

This notebook summarizes the full-span top-5 reruns for the Classical ML plus quantum branches branch study.

The main outputs are:

- setup-level rankings averaged across the four indices
- case-level leaderboards for each `index x horizon` case
- tuning-versus-pure-test scatterplots for the rerun top-5 configs
- branch deltas relative to the conventional baseline for matched configs


In [ ]:

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def resolve_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Run this notebook from the project root or its notebooks directory.')


PROJECT_ROOT = resolve_project_root()
SOURCE_ROOT = PROJECT_ROOT / 'outputs' / '11_major_index_ml_top5_full_branch_runs'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / '13_major_index_ml_top5_branch_comparison'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR = OUTPUT_ROOT / 'figures'
for directory in [OUTPUT_ROOT, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SCORES_PATH = SOURCE_ROOT / 'tables' / 'scores_summary.csv'
SETUP_SOURCE_PATH = SOURCE_ROOT / 'tables' / 'setup_summary.csv'
CASE_SOURCE_PATH = SOURCE_ROOT / 'tables' / 'case_summary.csv'
SETUP_RANKING_PATH = TABLE_DIR / 'setup_ranking.csv'
CASE_RANKING_PATH = TABLE_DIR / 'case_ranking.csv'
BRANCH_DELTA_PATH = TABLE_DIR / 'branch_delta_vs_conventional.csv'
BEST_SETUP_PATH = TABLE_DIR / 'best_branch_config_by_setup.csv'
BEST_CASE_PATH = TABLE_DIR / 'best_branch_config_by_case.csv'


In [ ]:

scores = pd.read_csv(SCORES_PATH)
setup = pd.read_csv(SETUP_SOURCE_PATH)
case = pd.read_csv(CASE_SOURCE_PATH)

setup_ranking = setup.sort_values(
    ['model_name', 'window_size', 'avg_pure_test_abs_error_pct_mean', 'avg_pure_test_close_RMSE', 'branch_name', 'config_rank']
).copy().reset_index(drop=True)
setup_ranking['setup_overall_rank'] = setup_ranking.groupby(['model_name', 'window_size']).cumcount() + 1
best_setup = setup_ranking.groupby(['model_name', 'window_size'], as_index=False).first()

case_ranking = case.sort_values(
    ['case_key', 'pure_test_abs_error_pct_mean', 'pure_test_close_RMSE', 'branch_name', 'model_name', 'config_rank']
).copy().reset_index(drop=True)
best_case = case_ranking.groupby('case_key', as_index=False).first()

conv = setup_ranking.loc[setup_ranking['branch_name'].eq('conventional')].copy()
quant = setup_ranking.loc[~setup_ranking['branch_name'].eq('conventional')].copy()
branch_delta = quant.merge(
    conv[
        ['model_name', 'window_size', 'config_key', 'avg_pure_test_abs_error_pct_mean', 'avg_pure_test_close_RMSE']
    ].rename(
        columns={
            'avg_pure_test_abs_error_pct_mean': 'conventional_avg_pure_test_abs_error_pct_mean',
            'avg_pure_test_close_RMSE': 'conventional_avg_pure_test_close_RMSE',
        }
    ),
    on=['model_name', 'window_size', 'config_key'],
    how='left',
)
branch_delta['delta_abs_error_pct_mean'] = branch_delta['avg_pure_test_abs_error_pct_mean'] - branch_delta['conventional_avg_pure_test_abs_error_pct_mean']
branch_delta['delta_close_RMSE'] = branch_delta['avg_pure_test_close_RMSE'] - branch_delta['conventional_avg_pure_test_close_RMSE']

setup_ranking.to_csv(SETUP_RANKING_PATH, index=False)
case_ranking.to_csv(CASE_RANKING_PATH, index=False)
branch_delta.to_csv(BRANCH_DELTA_PATH, index=False)
best_setup.to_csv(BEST_SETUP_PATH, index=False)
best_case.to_csv(BEST_CASE_PATH, index=False)

print('Best setup-level configs:')
display(best_setup[['model_name', 'window_size', 'branch_name', 'config_rank', 'effective_config_summary', 'avg_pure_test_abs_error_pct_mean', 'avg_pure_test_close_RMSE']])
print('Best case-level configs:')
display(best_case[['case_key', 'model_name', 'branch_name', 'config_rank', 'effective_config_summary', 'pure_test_abs_error_pct_mean', 'pure_test_close_RMSE']])


In [ ]:

def plot_tuning_vs_test(setup_ranking: pd.DataFrame, out_path: Path) -> None:
    models = list(setup_ranking['model_name'].drop_duplicates())
    windows = sorted(setup_ranking['window_size'].drop_duplicates())
    fig, axes = plt.subplots(len(models), len(windows), figsize=(5.0 * len(windows), 3.8 * len(models)), squeeze=False)
    palette = {
        'conventional': '#233049',
        'quantum_ideal': '#0E7C7B',
        'quantum_aer_noisy': '#2F5496',
        'quantum_mthree_readout': '#B89550',
        'quantum_zne': '#9C3B36',
    }
    for i, model_name in enumerate(models):
        for j, window_size in enumerate(windows):
            ax = axes[i][j]
            subset = setup_ranking[(setup_ranking['model_name'] == model_name) & (setup_ranking['window_size'] == window_size)].copy()
            for branch_name, block in subset.groupby('branch_name'):
                ax.scatter(
                    block['avg_tuning_abs_error_pct_mean'],
                    block['avg_pure_test_abs_error_pct_mean'],
                    s=52,
                    c=palette.get(branch_name, '#A7B3C4'),
                    label=branch_name,
                    alpha=0.9,
                    edgecolor='white',
                    linewidth=0.4,
                )
                for _, row in block.iterrows():
                    ax.text(row['avg_tuning_abs_error_pct_mean'], row['avg_pure_test_abs_error_pct_mean'], str(int(row['config_rank'])), fontsize=7, ha='center', va='center', color='white')
            ax.set_title(f'{model_name} | w={window_size}')
            ax.set_xlabel('Avg tuning abs % error')
            ax.set_ylabel('Avg pure-test abs % error')
            ax.grid(alpha=0.25)
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper center', ncol=5, frameon=False)
    fig.suptitle('Classical ML plus quantum branches: top-5 rerun configs, tuning vs pure-test', fontsize=14, y=1.03)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.close(fig)


def plot_branch_deltas(branch_delta: pd.DataFrame, out_path: Path) -> None:
    if branch_delta.empty:
        return
    grouped = branch_delta.groupby('branch_name', as_index=False)['delta_abs_error_pct_mean'].mean().sort_values('delta_abs_error_pct_mean')
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    ax.bar(grouped['branch_name'], grouped['delta_abs_error_pct_mean'], color=['#2F5496', '#B89550', '#9C3B36', '#0E7C7B'])
    ax.axhline(0.0, color='black', linewidth=0.8)
    ax.set_ylabel('Delta pure-test abs % error vs conventional')
    ax.set_title('Classical ML plus quantum branches: average branch delta relative to conventional')
    ax.tick_params(axis='x', rotation=25)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.close(fig)


plot_tuning_vs_test(setup_ranking, FIG_DIR / '01_tuning_vs_pure_test_scatter.png')
plot_branch_deltas(branch_delta, FIG_DIR / '02_branch_delta_vs_conventional.png')
print('Saved figures:')
for path in sorted(FIG_DIR.glob('*.png')):
    print('-', path.name)
